Function: synchronize IMU + odometry onto one timeline.

"""
Resample ACC, GYRO, ODO to a common sample rate using interpolation. (According to QC)

Input: Labeled runs from data/labeled/ (or raw runs from data/raw/)
Output: Resampled runs with uniform dt for all sensors

Strategy:
1. Determine target sample rate (e.g., 100 Hz based on your QC pass criteria)
2. Create unified time grid from min(t_rel) to max(t_rel)
3. Interpolate each sensor to the grid (linear or nearest-neighbor)
4. Preserve label column during resampling
5. Save to data/resampled/
"""


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from dataclasses import replace

# Make project root importable
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, save_labeled_run, discover_run_dirs

In [2]:
# Load all labeled runs from data/labeled/
labeled_data_dir = PROJECT_ROOT / "data" / "labeled"
labeled_run_dirs = sorted(labeled_data_dir.glob("log_*"))

labeled_runs = {}
for run_dir in labeled_run_dirs:
    print(f"Loading {run_dir.name}...")
    try:
        run = load_run(run_dir, include_pose=False)  # Exclude pose
        labeled_runs[run.run_id] = run
        print(f"Loaded {run.run_id}: ACC={len(run.acc)}, GYRO={len(run.gyro)}, ODO={len(run.odo)}")
    except Exception as e:
        print(f"Failed to load {run_dir.name}: {e}")

print(f"\nTotal labeled runs: {len(labeled_runs)}")

Loading log_20260216_114652.530...


/Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/src/io.py:61: DtypeWarning: Columns (0: label) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded log_20260216_114652.530: ACC=145708, GYRO=147427, ODO=219888
Loading log_20260223_142511.490...
Loaded log_20260223_142511.490: ACC=243359, GYRO=243378, ODO=365121

Total labeled runs: 2


In [ ]:
def resample_sensors(
    acc: pd.DataFrame,
    gyro: pd.DataFrame,
    odo: pd.DataFrame,
    target_hz: float = 100.0,
    tcol: str = "t_rel",
    method: str = "linear",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Resample accelerometer, gyroscope, and odometry to a common sample rate.
    
    Creates a unified time grid and interpolates all sensors to it.
    Preserves label column if present.
    
    Args:
        acc: Accelerometer DataFrame
        gyro: Gyroscope DataFrame  
        odo: Odometry DataFrame
        target_hz: Target sample rate in Hz (default: 100)
        tcol: Time column name (default: 't_rel')
        method: Interpolation method - 'linear' or 'nearest'
        
    Returns:
        Tuple of (acc_resampled, gyro_resampled, odo_resampled)
    """
    # Determine the unified time grid
    t_min = min(acc[tcol].min(), gyro[tcol].min(), odo[tcol].min())
    t_max = max(acc[tcol].max(), gyro[tcol].max(), odo[tcol].max())
    dt = 1.0 / target_hz
    
    # Create unified time grid
    unified_time = np.arange(t_min, t_max + dt / 2, dt)
    
    def interpolate_sensor(df: pd.DataFrame, new_time: np.ndarray, tcol: str, method: str) -> pd.DataFrame:
        """Interpolate a sensor DataFrame to new time grid, preserving labels."""
        label_col = "label" if "label" in df.columns else None
        numeric_cols = [c for c in df.columns if c != tcol and c != label_col]
        
        # Interpolate numeric columns
        data_resampled = {}
        for col in numeric_cols:
            if method == "linear":
                data_resampled[col] = np.interp(new_time, df[tcol], df[col])
            elif method == "nearest":
                indices = np.searchsorted(df[tcol], new_time)
                indices = np.clip(indices, 0, len(df) - 1)
                data_resampled[col] = df[col].iloc[indices].values
            else:
                raise ValueError(f"Unknown method: {method}")
        
        # Forward-fill labels (preserve prior labels for resampled points)
        if label_col:
            label_indices = np.searchsorted(df[tcol], new_time, side="right") - 1
            label_indices = np.clip(label_indices, 0, len(df) - 1)
            data_resampled[label_col] = df[label_col].iloc[label_indices].values
        
        # Create resampled DataFrame
        result = pd.DataFrame(data_resampled)
        result.insert(0, tcol, new_time)
        return result.reset_index(drop=True)
    
    # Resample all sensors
    acc_resampled = interpolate_sensor(acc, unified_time, tcol, method)
    gyro_resampled = interpolate_sensor(gyro, unified_time, tcol, method)
    odo_resampled = interpolate_sensor(odo, unified_time, tcol, method)
    
    return acc_resampled, gyro_resampled, odo_resampled

In [4]:
# Resample all labeled runs to 100 Hz
resampled_runs = {}
target_hz = 100.0

for run_id, run in labeled_runs.items():
    acc_rs, gyro_rs, odo_rs = resample_sensors(
        run.acc, run.gyro, run.odo,
        target_hz=target_hz,
        tcol="t_rel",
        method="linear"
    )
    
    # Create resampled RunData
    run_resampled = replace(run, acc=acc_rs, gyro=gyro_rs, odo=odo_rs, pose=None)
    resampled_runs[run_id] = run_resampled
    
    print(f"\n{run_id} (resampled to {target_hz} Hz):")
    print(f"  ACC: {len(run.acc)} → {len(acc_rs)} samples")
    print(f"  GYRO: {len(run.gyro)} → {len(gyro_rs)} samples")
    print(f"  ODO: {len(run.odo)} → {len(odo_rs)} samples")


log_20260216_114652.530 (resampled to 100.0 Hz):
  ACC: 145708 → 176947 samples
  GYRO: 147427 → 176947 samples
  ODO: 219888 → 176947 samples

log_20260223_142511.490 (resampled to 100.0 Hz):
  ACC: 243359 → 292115 samples
  GYRO: 243378 → 292115 samples
  ODO: 365121 → 292115 samples


# Save resampled runs to data/resampled/
output_root = PROJECT_ROOT / "data" / "resampled"

for run_id, run in resampled_runs.items():
    save_labeled_run(run, output_root)
    print(f"Saved {run_id} to {output_root / run_id}")
    
    for f in sorted((output_root / run_id).glob("*.csv")):
        file_lines = len(pd.read_csv(f))
        print(f"  {f.name}: {file_lines} rows")

from src.preprocess import qc_report

# Generate QC report for resampled runs
for run_id, run in resampled_runs.items():
    report = qc_report(run.sensors(), tcol="t_rel")
    print(f"\n{run_id} QC Report:")
    print(report.to_string())